# 🧠 v2 모델 실습 — 조건부 로짓 이해하기

**커널: Python (pipeline uv 환경)** (`cd pipeline && uv run --with jupyterlab jupyter lab --notebook-dir=..`)

## 수식 한 줄
경주 안에서 말 $i$의 우승확률을 $P(i) = \dfrac{e^{\beta \cdot x_i}}{\sum_j e^{\beta \cdot x_j}}$ 로 두고,
실제 우승마의 로그우도를 최대화해 $\beta$를 학습합니다 (경주 내 softmax = conditional logit).
$x$는 v1과 같은 7개 피처를 경주 내 min-max 정규화한 값(결측은 0.5 중립 대치)입니다.

In [1]:
# 학습된 계수 읽기 — 데이터가 말해주는 피처 중요도
import json
from kra_predict.score import MODEL_PATH

doc = json.loads(MODEL_PATH.read_text("utf-8"))
print(f"{doc['version']} · 학습 {doc['trainFrom']}~{doc['trainTo']} ({doc['trainRaces']}경주, L2={doc['l2']})")
for name, beta in sorted(doc["beta"].items(), key=lambda kv: -abs(kv[1])):
    bar = "█" * max(1, round(abs(beta) * 20))
    print(f"  {name:22s} {beta:+.3f} {bar}")

v2 · 학습 2025-07~2026-06 (2129경주, L2=0.01)
  placeRate1y            +1.202 ████████████████████████
  jockeyWinRate          +0.795 ████████████████
  winRate1y              +0.580 ████████████
  trainerWinRate         +0.407 ████████
  rating                 +0.253 █████
  rest                   +0.104 ██
  bodyWeightStability    +0.036 █


> 관찰 포인트: v1은 승률·레이팅 중심으로 **사람이 정한** 가중치였지만, 데이터는 **입상률(placeRate1y)과 기수**를 더 중시합니다.

In [2]:
# 같은 경주를 v1 / v2로 각각 예측해 비교
from unittest import mock
from kra_predict import score
from kra_predict.api.client import KraClient
from kra_predict.fetch import fetch_meet_bundle
from kra_predict.features import assemble_races

client = KraClient(offline=True)
bundle = fetch_meet_bundle(client, "2026-08-15")
client.close()
race = assemble_races(bundle)[0]

pred_v2 = score.build_prediction(race["entries"], race_date=race["date"], generated_at="lab")
with mock.patch.object(score, "load_learned_model", lambda: None):
    pred_v1 = score.build_prediction(race["entries"], race_date=race["date"], generated_at="lab")

print(f"{'마번':>4} {'v1 winProb':>11} {'v2 winProb':>11}")
v1p = {r["gateNo"]: r["winProb"] for r in pred_v1["rankings"]}
for r in sorted(pred_v2["rankings"], key=lambda r: r["gateNo"]):
    print(f"{r['gateNo']:>4} {v1p[r['gateNo']]:>10.1%} {r['winProb']:>10.1%}")
print("v1 픽:", pred_v1["topPicks"]["win"], "| v2 픽:", pred_v2["topPicks"]["win"])

픽스처 없음 → 빈 응답 처리: chulmainfo/numOfRows=100_pageNo=1_race_dt=20260815_rccrs_cd=1.json


픽스처 없음 → 빈 응답 처리: chulmainfo/numOfRows=100_pageNo=1_race_dt=20260815_rccrs_cd=3.json


픽스처 없음 → 빈 응답 처리: textDataHoldBuPtinInfo/numOfRows=100_pageNo=1_race_dt=20260815.json


픽스처 없음 → 빈 응답 처리: Jockey_Change_Detail/meet=3_numOfRows=100_pageNo=1_rc_date=20260815.json


픽스처 없음 → 빈 응답 처리: Jockey_Change_Detail/meet=2_numOfRows=100_pageNo=1_rc_date=20260815.json


...

픽스처 없음 → 빈 응답 처리: textDataHoldSeRaceInfo/numOfRows=100_pageNo=1_race_dt=20260815.json


픽스처 없음 → 빈 응답 처리: textDataHoldBuRaceInfo/numOfRows=100_pageNo=1_race_dt=20260815.json


픽스처 없음 → 빈 응답 처리: Race_Result_total/meet=3_numOfRows=100_pageNo=1_rc_date=20260815.json


픽스처 없음 → 빈 응답 처리: Race_Result_total/meet=2_numOfRows=100_pageNo=1_rc_date=20260815.json


  마번  v1 winProb  v2 winProb
   1      13.4%      16.2%
   2      44.9%      34.1%
   3       2.1%       4.9%
   4      23.1%      16.8%
   5       5.2%       7.5%
   6       2.2%       6.4%
   7       9.1%      14.1%
v1 픽: 2 | v2 픽: 2


In [3]:
# 미니 학습 실습 — 백테스트 캐시가 있으면 두 달치로 β를 직접 학습해 본다 (저장 안 함)
try:
    from kra_predict import config
    from kra_predict.api.client import KraClient
    from kra_predict.backtest import build_backtest_races, fetch_detail_rows, month_list_with_history
    from kra_predict.train import build_training_blocks, evaluate, fit_conditional_logit, FEATURES

    months = ["2026-05", "2026-06"]
    client = KraClient(config.service_key())
    rows = fetch_detail_rows(client, month_list_with_history(months))  # 캐시 히트면 무비용
    client.close()
    blocks = build_training_blocks(build_backtest_races(rows, months))
    beta = fit_conditional_logit(blocks, l2=0.01)
    print(f"{len(blocks)}경주로 학습한 β (참고용 — 저장하지 않음):")
    for name, b in zip(FEATURES, beta):
        print(f"  {name:22s} {b:+.3f}")
    print("인샘플:", evaluate(blocks, beta))
except Exception as e:  # .env/캐시 없으면 건너뜀
    print("건너뜀:", e)

순위 기록이 불완전한 경주 71건 제외 (평가 388건)


388경주로 학습한 β (참고용 — 저장하지 않음):
  winRate1y              +0.572
  placeRate1y            +1.163
  rating                 +0.165
  jockeyWinRate          +0.638
  trainerWinRate         +0.501
  bodyWeightStability    +0.089
  rest                   +0.097
인샘플: {'races': 388, 'logLoss': 2.0115, 'winRate': 0.3144}


## 다음 실험 아이디어 (이슈 #24)
- 피처 추가: 거리 적성, 배당(시장 정보) — 단, 배당은 발주 직전에야 확정되는 점 주의
- LightGBM(LambdaRank)으로 비선형 상호작용 — v2 대비 log-loss 개선 시에만 채택
- 확률 캘리브레이션(isotonic) — winProb를 실제 빈도에 맞추기
- **철칙**: 학습/평가 기간 분리, 백테스트 하네스의 as-of 원칙 유지